# Deploying scikit-learn and XGBoost models on TrueFoundry, without writing inference code

Register the model once — TrueFoundry generates the inference code, requirements and Dockerfile.

- No Seldon `model-settings.json`, no server wrapper, no hand-written FastAPI per model
- One flow for both frameworks, differences handled by the logged schema
- You own the Dockerfile, python version and dependencies, patched and scanned on your schedule
- Generated code is editable, testable locally, committable to your repo

**Three things must be logged or the model is not deployable:**

1. model file path
2. serialization format
3. schema, from `sklearn_infer_schema` / `xgboost_infer_schema`

The schema is what generates correct inference code, including which method to call.

**Prerequisites:** TrueFoundry CLI logged in (`tfy login --host https://<your-control-plane>.truefoundry.cloud`), an ML repo and a workspace.

📄 [Model deployment without writing inference code with TrueFoundry](https://truefoundry.atlassian.net/wiki/spaces/~712020cdcb3fede2124d2bb5913e77d2bd20ab/pages/1785528378/Model+deployment+without+writing+inference+code+with+TrueFoundry)

In [ ]:
%pip install -q truefoundry scikit-learn xgboost joblib

## 1. Config

In [ ]:
ML_REPO = "my-ml-repo"                  # existing ML repo

SK_FILE = "iris-sklearn.joblib"
XGB_FILE = "iris-xgboost.joblib"

## 2. Train and save

Both models land as plain `.joblib` files next to this notebook. Swap in your own.

In [ ]:
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

X, y = load_iris(return_X_y=True)

sk_model = RandomForestClassifier(n_estimators=10).fit(X, y)
joblib.dump(sk_model, SK_FILE)

xgb_model = XGBClassifier(n_estimators=10).fit(X, y)
joblib.dump(xgb_model, XGB_FILE)

## 3. Log to the model registry

`model_schema` is what makes the model deployable — it records the input/output shape and
the inference method, and that is what the generated inference code is built from.

In [ ]:
from truefoundry.ml import (
    get_client,
    SklearnFramework, sklearn_infer_schema,
    XGBoostFramework, xgboost_infer_schema,
)

client = get_client()

In [ ]:
sk_version = client.log_model(
    ml_repo=ML_REPO,
    name="iris-sklearn",
    model_file_or_folder=SK_FILE,
    framework=SklearnFramework(
        model_filepath=SK_FILE,
        serialization_format="joblib",
        model_schema=sklearn_infer_schema(
            model_input=X, model=sk_model, infer_method_name="predict",
        ),
    ),
)
sk_version.fqn

In [ ]:
# XGBoost only supports the `predict` inference method, so there is no
# infer_method_name argument here.
xgb_version = client.log_model(
    ml_repo=ML_REPO,
    name="iris-xgboost",
    model_file_or_folder=XGB_FILE,
    framework=XGBoostFramework(
        model_filepath=XGB_FILE,
        serialization_format="joblib",
        model_schema=xgboost_infer_schema(model_input=X, model=xgb_model),
    ),
)
xgb_version.fqn

## 4. Deploy

In the TrueFoundry UI: **ML Repos → your repo → the model version → Deploy**, pick a workspace.

TrueFoundry generates the FastAPI inference code, `requirements.txt` and `Dockerfile` from the
logged schema. The generated code is yours — edit it, test it locally, commit it to your repo.

📄 [Model deployment without writing inference code with TrueFoundry](https://truefoundry.atlassian.net/wiki/spaces/~712020cdcb3fede2124d2bb5913e77d2bd20ab/pages/1785528378/Model+deployment+without+writing+inference+code+with+TrueFoundry)